In [1]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas - kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas - kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as _sum, col, desc

spark = SparkSession.builder.appName("Tugas5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

df_transaksi.show(5)

+--------+--------------------+----------+------------+------------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|
+--------+--------------------+----------+------------+------------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|
+--------+--------------------+----------+------------+------------+
only showing top 5 rows



In [5]:
# 1. Hitung total pendapatan per kota (Pendapatan = unit_terjual * harga_satuan)
df_pendapatan_kota = df_transaksi \
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
    .groupBy("kota") \
    .agg(_sum("total_pendapatan").alias("total_pendapatan"))

# 2. Join dengan df_target dan hitung persentase pencapaian
df_pencapaian = df_pendapatan_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100) \
    .orderBy(desc("pencapaian_persen"))

df_pencapaian.show(truncate=False)

[Stage 6:>                                                          (0 + 4) / 4]

+----------+----------------+--------------+----------+------------------+
|kota      |total_pendapatan|target_bulanan|pic_cabang|pencapaian_persen |
+----------+----------------+--------------+----------+------------------+
|Purworejo |45650000        |30000000      |Fitri     |152.16666666666669|
|Solo      |33475000        |40000000      |Bayu      |83.6875           |
|Yogyakarta|47275000        |60000000      |Joko      |78.79166666666667 |
|Magelang  |31650000        |45000000      |Rani      |70.33333333333334 |
|Semarang  |38175000        |55000000      |Sari      |69.4090909090909  |
+----------+----------------+--------------+----------+------------------+



In [7]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, desc

# 1. Hitung total pendapatan per kota dan kategori terlebih dahulu
df_kat_kota = df_transaksi \
    .withColumn("pendapatan", col("unit_terjual") * col("harga_satuan")) \
    .groupBy("kota", "kategori") \
    .agg(_sum("pendapatan").alias("total_pendapatan"))

# 2. Terapkan Window Function dengan row_number() dipartisi per kota dan diurutkan pendapatan tertinggi
window_spec = Window.partitionBy("kota").orderBy(desc("total_pendapatan"))

df_top_kategori = df_kat_kota.withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn") \
    .orderBy("kota")

df_top_kategori.show(truncate=False)

+----------+----------------------+----------------+
|kota      |kategori              |total_pendapatan|
+----------+----------------------+----------------+
|Magelang  |Kesehatan & Kecantikan|7275000         |
|Purworejo |Kesehatan & Kecantikan|10075000        |
|Semarang  |Rumah Tangga          |11125000        |
|Solo      |Kesehatan & Kecantikan|8425000         |
|Yogyakarta|Fashion               |13325000        |
+----------+----------------------+----------------+



In [8]:
# Daftarkan ke temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Jalankan kueri Spark SQL
query_sql = """
    t.kota, 
    t.pic_cabang, 
    COUNT(d.order_id) AS jumlah_transaksi
    FROM target t
    LEFT JOIN transaksi d ON t.kota = d.kota
    GROUP BY t.kota, t.pic_cabang
    ORDER BY jumlah_transaksi DESC
"""
# Catatan: Jika ingin menggunakan INNER JOIN atau LEFT JOIN sesuaikan kebutuhan, di sini menggunakan LEFT JOIN agar cabang yang transaksinya kosong tetap tercatat.
# Mari gunakan query yang bersih:
hasil_sql = spark.sql("""
    SELECT t.kota, t.pic_cabang, COUNT(d.order_id) AS jumlah_transaksi
    FROM target t
    LEFT JOIN transaksi d ON t.kota = d.kota
    GROUP BY t.kota, t.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

hasil_sql.show(truncate=False)

[Stage 14:=============================>                            (2 + 2) / 4]

+----------+----------+----------------+
|kota      |pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
|Purworejo |Fitri     |116             |
|Yogyakarta|Joko      |110             |
|Solo      |Bayu      |95              |
|Semarang  |Sari      |93              |
|Magelang  |Rani      |86              |
+----------+----------+----------------+

